In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [47]:
import pandas as pd
import numpy as np
import os
import torch
from transformers import BartForConditionalGeneration, BartTokenizer

print(f"GPU disponibil: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
    print(f"Memorie GPU: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
import transformers
print(f"Transformers version: {transformers.__version__}")

GPU disponibil: True
Device: Tesla T4
Memorie GPU: 14.6 GB
Transformers version: 5.0.0


In [48]:
INPUT_PATH = '/kaggle/input/datasets/catalinalupu/movies-ready-for-summarization/movies-ready-for-summarization.csv'
OUTPUT_PATH = '/kaggle/working/movies_with_summaries.csv'

df = pd.read_csv(INPUT_PATH)
print(f"Filme incarcate: {len(df):,}")
print(f"Coloane: {df.columns.tolist()}")
print(f"\nExemplu input_text:")
print(df['input_text'].iloc[0])

Filme incarcate: 40,197
Coloane: ['movie_id', 'title', 'overview', 'input_text', 'genres', 'keywords', 'director', 'release_date', 'runtime', 'popularity', 'tagline', 'review_texts', 'certification', 'cast', 'has_review']

Exemplu input_text:
Toy Story. Led by Woody, Andys toys live happily in his room until Andys birthday brings Buzz Lightyear onto the scene. Afraid of losing his place in Andys heart, Woody plots against Buzz. But when circumstances separate Buzz and Woody from their owner, the duo eventually learns to put aside their differences.


In [49]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

MODEL_NAME = "facebook/bart-large-cnn"
tokenizer = BartTokenizer.from_pretrained(MODEL_NAME)
model = BartForConditionalGeneration.from_pretrained(MODEL_NAME).to(device)
model.eval()

print("Model incarcat")

test_text = "Inception. Dom Cobb is a skilled thief, the absolute best in the dangerous art of extraction — stealing valuable secrets from deep within the subconscious during the dream state, when the mind is at its most vulnerable. Cobb's rare ability has made him a coveted player in this treacherous new world of corporate espionage, but it has also made him an international fugitive and cost him everything he has ever loved. Now Cobb is being offered a chance at redemption. One last job could give him his life back but only if he can accomplish the impossible inception. Instead of the perfect heist, Cobb and his team of specialists have to pull off the reverse — their task is not to steal an idea, but to plant one. If they succeed, it could be the perfect crime. But no amount of careful planning or expertise can prepare the team for the dangerous enemy that seems to predict their every move."
inputs = tokenizer(
    test_text,
    return_tensors="pt",
    max_length=512,
    truncation=True
).to(device)

with torch.no_grad():
    summary_ids = model.generate(
        inputs["input_ids"],
        max_length = 40,
        min_length = 15,
        num_beams = 4,
        early_stopping=True,
        length_penalty=2.0,        
        no_repeat_ngram_size=3,    
        forced_bos_token_id=0
    )

result = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
print(f"\nTest:")
print(f"Input: {test_text}")
print(f"Output: {result}")

Device: cuda


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

Model incarcat

Test:
Input: Inception. Dom Cobb is a skilled thief, the absolute best in the dangerous art of extraction — stealing valuable secrets from deep within the subconscious during the dream state, when the mind is at its most vulnerable. Cobb's rare ability has made him a coveted player in this treacherous new world of corporate espionage, but it has also made him an international fugitive and cost him everything he has ever loved. Now Cobb is being offered a chance at redemption. One last job could give him his life back but only if he can accomplish the impossible inception. Instead of the perfect heist, Cobb and his team of specialists have to pull off the reverse — their task is not to steal an idea, but to plant one. If they succeed, it could be the perfect crime. But no amount of careful planning or expertise can prepare the team for the dangerous enemy that seems to predict their every move.
Output: Dom Cobb is a skilled thief, the absolute best in the dangerous art o

In [50]:
sample_lengths = []
for text in df['input_text'].dropna().sample(1000, random_state=42):
    tokens = tokenizer(text, truncation=False)['input_ids']
    sample_lengths.append(len(tokens))

import numpy as np
print(f"Tokeni — Medie:  {np.mean(sample_lengths):.1f}")
print(f"Tokeni — Maxim:  {max(sample_lengths)}")
print(f"Tokeni — 99%:    {np.percentile(sample_lengths, 99):.1f}")
print(f"\nFilme care AR FI trunchiate la 512 tokeni: "
      f"{sum(l > 512 for l in sample_lengths)}/1000")

Tokeni — Medie:  63.7
Tokeni — Maxim:  221
Tokeni — 99%:    180.0

Filme care AR FI trunchiate la 512 tokeni: 0/1000


In [52]:
def summarize_batch(texts, max_len=40, min_len=15):
    results = []
    valid_indices = []
    valid_texts = []

    for i, text in enumerate(texts):
        if not isinstance(text, str) or len(text.split()) < 10:
            results.append((i, text if isinstance(text,str) else ""))
        else:
            valid_indices.append(i)
            valid_texts.append(text)

    if valid_texts:
        try:
            inputs = tokenizer(
                valid_texts,
                return_tensors="pt",
                max_length=512,
                truncation=True,
                padding=True
            ).to(device)

            with torch.no_grad():
                summary_ids = model.generate(
                    inputs["input_ids"],
                    attention_mask=inputs["attention_mask"],
                    max_length=max_len,
                    min_length=min_len,
                    num_beams=4,
                    early_stopping=True,
                    length_penalty=2.0,     
                    no_repeat_ngram_size=3, 
                    forced_bos_token_id=0 
                )

            summaries = [
                tokenizer.decode(s, skip_special_tokens=True)
                for s in summary_ids
            ]

            for idx, summary in zip(valid_indices, summaries):
                results.append((idx, summary))

        except Exception as e:
            print(f"Eroare batch: {e}")
            for idx, text in zip(valid_indices, valid_texts):
                results.append((idx, text))

    results.sort(key=lambda x: x[0])
    return [r[1] for r in results]

In [53]:
BATCH_SIZE = 16

if os.path.exists(OUTPUT_PATH):
    done_df = pd.read_csv(OUTPUT_PATH)
    start_idx = len(done_df)
    print(f"Checkpoint gasit! Reluare de la filmul {start_idx}/{len(df)}")
else:
    done_df = pd.DataFrame()
    start_idx = 0
    print(f"Start de la 0 - {len(df):,} filme de procesat")

all_batches = [done_df] if len(done_df) > 0 else []

for i in range(start_idx, len(df), BATCH_SIZE):
    batch = df.iloc[i:i+BATCH_SIZE].copy()
    texts = batch['input_text'].tolist()

    batch['overview_summary'] = summarize_batch(texts)
    all_batches.append(batch)

    pd.concat(all_batches, ignore_index=True).to_csv(OUTPUT_PATH, index=False)

    processed = min(i+ BATCH_SIZE, len(df))
    pct = processed / len(df) * 100
    print(f"[{pct:5.1f}%] {processed:,}/{len(df):,} filme procesate")

print("\n Sumarizare completa!")

Start de la 0 - 40,197 filme de procesat
[  0.0%] 16/40,197 filme procesate
[  0.1%] 32/40,197 filme procesate
[  0.1%] 48/40,197 filme procesate
[  0.2%] 64/40,197 filme procesate
[  0.2%] 80/40,197 filme procesate
[  0.2%] 96/40,197 filme procesate
[  0.3%] 112/40,197 filme procesate
[  0.3%] 128/40,197 filme procesate
[  0.4%] 144/40,197 filme procesate
[  0.4%] 160/40,197 filme procesate
[  0.4%] 176/40,197 filme procesate
[  1.3%] 528/40,197 filme procesate
[  1.4%] 544/40,197 filme procesate
[  1.4%] 560/40,197 filme procesate
[  1.4%] 576/40,197 filme procesate
[  1.5%] 592/40,197 filme procesate
[  1.5%] 608/40,197 filme procesate
[  1.6%] 624/40,197 filme procesate
[  1.6%] 640/40,197 filme procesate
[  1.6%] 656/40,197 filme procesate
[  1.7%] 672/40,197 filme procesate
[  1.7%] 688/40,197 filme procesate
[  1.8%] 704/40,197 filme procesate
[  1.8%] 720/40,197 filme procesate
[  1.8%] 736/40,197 filme procesate
[  1.9%] 752/40,197 filme procesate
[  1.9%] 768/40,197 filme pro

In [54]:
final_df = pd.read_csv(OUTPUT_PATH)

print(f"Total filme procesate: {len(final_df):,}")
print(f"overview_summary null: {final_df['overview_summary'].isna().sum()}")
print(f"overview_summary gol: {(final_df['overview_summary'] == '').sum()}")

print("Exemple sumarizari:")
for _, row in final_df.sample(5, random_state=42).iterrows():
    print(f"\n {row['title']}")
    print(f" Original: {str(row['overview'])[:180]}")
    print(f" Sumarizat: {row['overview_summary']}")

print(f"\nFisier finalizat: {OUTPUT_PATH}")

Total filme procesate: 40,197
overview_summary null: 0
overview_summary gol: 0
Exemple sumarizari:

 Ballad of a Soldier
 Original: During World War II, earnest young Russian soldier Alyosha Skvortsov is rewarded with a short leave of absence for performing a heroic deed on the battlefield. Feeling homesick, he
 Sumarizat: Alyosha Skvortsov is rewarded with a short leave of absence for performing a heroic deed on the battlefield. Feeling homesick, he decides to visit his mother. Aly

 Pulimurugan
 Original: Murugan, with his tools and expertise, protects Puliyoors villagers from deadly tiger attacks. However, Daddy, an illegal drugs dealer, takes advantage of Murugans innocence and wr
 Sumarizat: Pulimurugan, with his tools and expertise, protects Puliyoors villagers from deadly tiger attacks. Daddy, an illegal drugs dealer, takes advantage of Murugans

 Afraid
 Original: Curtis Pike and his family are selected to test a new home device: a digital assistant called AIA. AIA observes the

Partea 2 - review 


In [13]:
import pandas as pd
import numpy as np
import os
import torch
from transformers import BartForConditionalGeneration, BartTokenizer

print(f"GPU disponibil: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
    print(f"Memorie GPU: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
import transformers
print(f"Transformers version: {transformers.__version__}")

GPU disponibil: True
Device: Tesla T4
Memorie GPU: 14.6 GB
Transformers version: 5.0.0


In [14]:
INPUT_PATH = '/kaggle/input/datasets/catalinalupu/movies-with-summaries/movies_with_summaries.csv'
OUTPUT_PATH = '/kaggle/working/movies_with_summaries.csv'

df = pd.read_csv(INPUT_PATH)
print(f"Filme incarcate: {len(df):,}")
print(f"Coloane: {df.columns.tolist()}")
print(f"\nFilme cu review: {df['has_review'].sum():,}")

Filme incarcate: 40,197
Coloane: ['movie_id', 'title', 'overview', 'input_text', 'genres', 'keywords', 'director', 'release_date', 'runtime', 'popularity', 'tagline', 'review_texts', 'certification', 'cast', 'has_review', 'overview_summary']

Filme cu review: 11,703


In [15]:
import ast

MAX_WORDS_INPUT = 500
 
def build_review_input(review_texts_raw):
    if not isinstance(review_texts_raw, str):
        return None
 
    try:
        reviews = ast.literal_eval(review_texts_raw)
    except Exception:
        return None
 
    if not isinstance(reviews, list) or len(reviews) == 0:
        return None
 
    cleaned = []
    for r in reviews:
        if isinstance(r, str) and len(r.split()) >= 5: 
            cleaned.append(r.strip())
 
    if not cleaned:
        return None
 
    combined = " ".join(cleaned)
 
    words = combined.split()
    if len(words) > MAX_WORDS_INPUT:
        combined = " ".join(words[:MAX_WORDS_INPUT])
 
    return combined
 
 
df['input_review'] = df['review_texts'].apply(build_review_input)
 
valid = df['input_review'].notna().sum()
print(f"\nFilme cu input_review valid: {valid:,}")
print(f"Filme fără input_review (vor primi ''): {len(df) - valid:,}")
 
sample = df[df['input_review'].notna()].iloc[0]
print(f"\nExemplu input_review pentru '{sample['title']}':")
print(sample['input_review'][:300], "...")
 


Filme cu input_review valid: 11,689
Filme fără input_review (vor primi ''): 28,508

Exemplu input_review pentru 'Toy Story':
This movie came out when I was three. Now I'm twenty seven and the goddamn thing still holds up. Final rating: - Very strong appeal. A personal favourite. Decided to revisit this after many years and still holds up so well. Great movie for both kids and adults with wonderful teachable moments. Just  ...


In [16]:
print(df.head(2)[['title', 'has_review', 'review_texts']].to_string())

       title  has_review                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                

In [17]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
 
MODEL_NAME = "facebook/bart-large-cnn"
tokenizer = BartTokenizer.from_pretrained(MODEL_NAME)
model = BartForConditionalGeneration.from_pretrained(MODEL_NAME).to(device)
model.eval()
print("Model încărcat.")

Device: cuda


vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

Model încărcat.


In [18]:
valid_reviews = df['input_review'].dropna().sample(min(500, valid), random_state=42)
token_lengths = []
for text in valid_reviews:
    tokens = tokenizer(text, truncation=False)['input_ids']
    token_lengths.append(len(tokens))
 
print(f"Tokeni per input_review:")
print(f"  Medie:      {np.mean(token_lengths):.1f}")
print(f"  Maxim:      {max(token_lengths)}")
print(f"  Percentila 99%: {np.percentile(token_lengths, 99):.1f}")
print(f"\nFilme care AR FI trunchiate la 512 tokeni: "
      f"{sum(l > 512 for l in token_lengths)}/{len(token_lengths)}"
      f" ({sum(l > 512 for l in token_lengths)/len(token_lengths)*100:.1f}%)")
 
test_title = df[df['input_review'].notna()].iloc[0]['title']
test_text  = df[df['input_review'].notna()].iloc[0]['input_review']
 
inputs = tokenizer(test_text, return_tensors="pt", max_length=512, truncation=True).to(device)
with torch.no_grad():
    ids = model.generate(
        inputs["input_ids"],
        max_length=60, min_length=15,
        num_beams=4, early_stopping=True,
        length_penalty=2.0, no_repeat_ngram_size=3,
        forced_bos_token_id=0
    )
print(f"\nTest sumarizare review pentru '{test_title}':")
print(f"Input (primele 200 chars): {test_text[:200]}...")
print(f"Summary: {tokenizer.decode(ids[0], skip_special_tokens=True)}")
 

Tokeni per input_review:
  Medie:      398.3
  Maxim:      757
  Percentila 99%: 702.0

Filme care AR FI trunchiate la 512 tokeni: 186/500 (37.2%)

Test sumarizare review pentru 'Toy Story':
Input (primele 200 chars): This movie came out when I was three. Now I'm twenty seven and the goddamn thing still holds up. Final rating: - Very strong appeal. A personal favourite. Decided to revisit this after many years and ...
Summary: Tom Hanks leads a strong cast. Tim Allen is great, too, as Buzz Lightyear. Don Rickles (Mr. Potato Head), Wallace Shawn (Rex) and John Ratzenberger (Hamm) also bring fun.


In [20]:
def summarize_reviews_batch(texts, max_len=60, min_len=15):
    results = [None] * len(texts)
    valid_indices = []
    valid_texts   = []
 
    for i, text in enumerate(texts):
        if not isinstance(text, str) or len(text.split()) < 10:
            results[i] = ""   
        else:
            valid_indices.append(i)
            valid_texts.append(text)
 
    if valid_texts:
        try:
            inputs = tokenizer(
                valid_texts,
                return_tensors="pt",
                max_length=512,
                truncation=True,
                padding=True
            ).to(device)
 
            with torch.no_grad():
                summary_ids = model.generate(
                    inputs["input_ids"],
                    attention_mask=inputs["attention_mask"],
                    max_length=max_len,
                    min_length=min_len,
                    num_beams=4,
                    early_stopping=True,
                    length_penalty=2.0,
                    no_repeat_ngram_size=3,
                    forced_bos_token_id=0
                )
 
            summaries = [
                tokenizer.decode(s, skip_special_tokens=True)
                for s in summary_ids
            ]
 
            for idx, summary in zip(valid_indices, summaries):
                results[idx] = summary
 
        except Exception as e:
            print(f"  Eroare batch: {e}")
            for idx, text in zip(valid_indices, valid_texts):
                results[idx] = text 
 
    return results
 

In [21]:
BATCH_SIZE = 16   
 
if os.path.exists(OUTPUT_PATH):
    done_df   = pd.read_csv(OUTPUT_PATH)
    start_idx = len(done_df)
    print(f"Checkpoint găsit! Reluare de la filmul {start_idx}/{len(df)}")
else:
    done_df   = pd.DataFrame()
    start_idx = 0
    print(f"Start de la 0 — {len(df):,} filme de procesat")
    print(f"(din care doar {valid:,} vor intra efectiv în BART)")
 
all_batches = [done_df] if len(done_df) > 0 else []
 
for i in range(start_idx, len(df), BATCH_SIZE):
    batch = df.iloc[i:i+BATCH_SIZE].copy()
 
    batch['review_summary'] = summarize_reviews_batch(
        batch['input_review'].tolist()
    )
 
    all_batches.append(batch)
    pd.concat(all_batches, ignore_index=True).to_csv(OUTPUT_PATH, index=False)
 
    processed = min(i + BATCH_SIZE, len(df))
    pct       = processed / len(df) * 100
    print(f"[{pct:5.1f}%] {processed:,}/{len(df):,} filme procesate")
 
print("\nSumarizare completă!")
 
 

Start de la 0 — 40,197 filme de procesat
(din care doar 11,689 vor intra efectiv în BART)
[  0.0%] 16/40,197 filme procesate
[  0.1%] 32/40,197 filme procesate
[  0.1%] 48/40,197 filme procesate
[  0.2%] 64/40,197 filme procesate
[  0.2%] 80/40,197 filme procesate
[  0.2%] 96/40,197 filme procesate
[  0.3%] 112/40,197 filme procesate
[  0.3%] 128/40,197 filme procesate
[  0.4%] 144/40,197 filme procesate
[  0.4%] 160/40,197 filme procesate
[  0.4%] 176/40,197 filme procesate
[  0.5%] 192/40,197 filme procesate
[  0.5%] 208/40,197 filme procesate
[  0.6%] 224/40,197 filme procesate
[  0.6%] 240/40,197 filme procesate
[  0.6%] 256/40,197 filme procesate
[  0.7%] 272/40,197 filme procesate
[  0.7%] 288/40,197 filme procesate
[  0.8%] 304/40,197 filme procesate
[  0.8%] 320/40,197 filme procesate
[  0.8%] 336/40,197 filme procesate
[  0.9%] 352/40,197 filme procesate
[  0.9%] 368/40,197 filme procesate
[  1.0%] 384/40,197 filme procesate
[  1.0%] 400/40,197 filme procesate
[  1.0%] 416/40,

In [22]:
final_df = pd.read_csv(OUTPUT_PATH)
print(f"Coloane finale: {final_df.columns.tolist()}")
print(f"\nReview summaries non-goale: {(final_df['review_summary'] != '').sum():,}")
print(f"Review summaries goale (fără recenzii): {(final_df['review_summary'] == '').sum():,}")
 
examples = final_df[final_df['review_summary'] != ''].head(3)
for _, row in examples.iterrows():
    print(f"\nFilm: {row['title']}")
    print(f"  Review input (primele 150 chars): {str(row['input_review'])[:150]}...")
    print(f"  Review summary: {row['review_summary']}")

Coloane finale: ['movie_id', 'title', 'overview', 'input_text', 'genres', 'keywords', 'director', 'release_date', 'runtime', 'popularity', 'tagline', 'review_texts', 'certification', 'cast', 'has_review', 'overview_summary', 'input_review', 'review_summary']

Review summaries non-goale: 40,197
Review summaries goale (fără recenzii): 0

Film: Toy Story
  Review input (primele 150 chars): This movie came out when I was three. Now I'm twenty seven and the goddamn thing still holds up. Final rating: - Very strong appeal. A personal favour...
  Review summary: Tom Hanks leads a strong cast. Tim Allen is great, too, as Buzz Lightyear. Don Rickles (Mr. Potato Head), Wallace Shawn (Rex) and John Ratzenberger (Hamm) also bring fun.

Film: Jumanji
  Review input (primele 150 chars): Throw the dice and take a turn, Jumanji made the critics gurn. Jumanji is directed by Joe Johnston and based on Chris Van Allsburg's short story of th...
  Review summary: Jumanji is based on Chris Van Allsburg's sho

In [26]:
df = pd.read_csv(OUTPUT_PATH)
print(df.columns.tolist())
print(f"\nTotal filme: {len(df):,}")
print(f"overview_summary non-goale: {df['overview_summary'].notna().sum():,}")
print(f"review_summary non-goale: {(df['review_summary'] != '').sum():,}")
print(f"tagline non-goale: {df['tagline'].notna().sum():,}")

sample = df[df['review_summary'] != ''].iloc[0]
print(f"\n {sample['title']} ")
print(f"Tagline:          {sample['tagline']}")
print(f"Overview summary: {sample['overview_summary']}")
print(f"Review summary:   {sample['review_summary']}")

['movie_id', 'title', 'overview', 'input_text', 'genres', 'keywords', 'director', 'release_date', 'runtime', 'popularity', 'tagline', 'review_texts', 'certification', 'cast', 'has_review', 'overview_summary', 'input_review', 'review_summary']

Total filme: 40,197
overview_summary non-goale: 40,197
review_summary non-goale: 40,197
tagline non-goale: 40,197

 Toy Story 
Tagline:          The adventure takes off when toys come to life!
Overview summary: Andys toys live happily in his room until Andys birthday brings Buzz Lightyear onto the scene. Afraid of losing his place in Andys heart, Woody plots against Buzz.
Review summary:   Tom Hanks leads a strong cast. Tim Allen is great, too, as Buzz Lightyear. Don Rickles (Mr. Potato Head), Wallace Shawn (Rex) and John Ratzenberger (Hamm) also bring fun.


In [4]:
import pandas as pd

df = pd.read_csv('/kaggle/input/datasets/catalinalupu/movies-with-review-summaries/movies_with_review_summaries.csv')
print(f"Filme încărcate: {len(df):,}")
print(f"Coloane existente: {df.columns.tolist()}")

df['title_year'] = df['title'] + ' (' + pd.to_datetime(df['release_date'], errors='coerce').dt.year.astype('Int64').astype(str) + ')'

dups = df['title_year'].duplicated().sum()
print(f"\nDuplicate title_year: {dups}")

if dups > 0:
    print(df[df['title_year'].duplicated(keep=False)][['movie_id','title_year','release_date']].head(10))

print("\nExemple title_year:")
print(df['title_year'].head(5).tolist())

Filme încărcate: 40,197
Coloane existente: ['movie_id', 'title', 'overview', 'input_text', 'genres', 'keywords', 'director', 'release_date', 'runtime', 'popularity', 'tagline', 'review_texts', 'certification', 'cast', 'has_review', 'overview_summary', 'input_review', 'review_summary']

Duplicate title_year: 38
       movie_id            title_year release_date
1810       9425        Soldier (1998)   1998-10-23
6560       1436    The Warrior (2001)   2001-09-07
7401      10943  The Gathering (2003)   2003-02-23
7403      23305    The Warrior (2001)   2001-09-23
7670       2026        Hostage (2005)   2005-03-10
8542       5289          Chaos (2005)   2005-01-17
9966      39914          Chaos (2005)   2005-08-10
12576     70585          Lucky (2011)   2011-07-15
12822     89691            ATM (2012)   2012-02-17
13646    191619        Beneath (2013)   2013-07-19

Exemple title_year:
['Toy Story (1995)', 'Jumanji (1995)', 'Grumpier Old Men (1995)', 'Waiting to Exhale (1995)', 'Father of t

In [5]:
dup_mask = df['title_year'].duplicated(keep=False)
dup_df = df[dup_mask][['movie_id', 'title_year', 'release_date', 'overview']].sort_values('title_year')
print(f"Total filme cu title_year duplicat: {len(dup_df)}")
print(dup_df.to_string())

Total filme cu title_year duplicat: 76
       movie_id             title_year release_date                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                             overview
31805    104651             ATM (2012)   2012-01-19                                                                                                                                                                     

In [6]:
df = pd.read_csv('/kaggle/input/datasets/catalinalupu/movies-with-review-summaries/movies_with_review_summaries.csv')
print(f"Filme încărcate: {len(df):,}")
print(f"Coloane existente: {df.columns.tolist()}")

Filme încărcate: 40,197
Coloane existente: ['movie_id', 'title', 'overview', 'input_text', 'genres', 'keywords', 'director', 'release_date', 'runtime', 'popularity', 'tagline', 'review_texts', 'certification', 'cast', 'has_review', 'overview_summary', 'input_review', 'review_summary']


In [7]:
!pip install sentence-transformers --quiet
 
import pandas as pd
import numpy as np
import torch
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import os
 
print(f"GPU disponibil: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")

GPU disponibil: True
Device: Tesla T4


In [8]:
INPUT_PATH  = '/kaggle/input/datasets/catalinalupu/movies-with-review-summaries/movies_with_review_summaries.csv'
OUTPUT_PATH = '/kaggle/working/'
 
df = pd.read_csv(INPUT_PATH)
print(f"Filme încărcate: {len(df):,}")
 
df['overview_summary'] = df['overview_summary'].fillna('')
df['review_summary']   = df['review_summary'].fillna('')
df['tagline']          = df['tagline'].fillna('')

log_pop  = np.log1p(df['popularity'])
df['popularity_norm'] = (log_pop - log_pop.min()) / (log_pop.max() - log_pop.min())
 
print(f"\nPopularitate normalizata:")
print(f"  Min: {df['popularity_norm'].min():.4f}")
print(f"  Max: {df['popularity_norm'].max():.4f}")
print(f"  Medie: {df['popularity_norm'].mean():.4f}")
print(f"  Filme cu pop=0 → {df.loc[df['popularity']==0, 'popularity_norm'].iloc[0]:.4f}")

has_review_mask = df['review_summary'] != ''
print(f"\nFilme cu review_summary: {has_review_mask.sum():,}")
print(f"Filme fara review_summary: {(~has_review_mask).sum():,}")
 
 

Filme încărcate: 40,197

Popularitate normalizata:
  Min: 0.0000
  Max: 1.0000
  Medie: 0.1729
  Filme cu pop=0 → 0.0000

Filme cu review_summary: 11,644
Filme fara review_summary: 28,553


In [9]:
MODEL_NAME = 'all-MiniLM-L6-v2'
device = 'cuda' if torch.cuda.is_available() else 'cpu'
 
sbert = SentenceTransformer(MODEL_NAME, device=device)
print(f"Model {MODEL_NAME} incarcat pe {device}")

test = sbert.encode(["A toy cowboy feels threatened by a new spaceman toy."])
print(f"Dimensiune embedding: {test.shape[1]}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model all-MiniLM-L6-v2 incarcat pe cuda
Dimensiune embedding: 384


In [10]:
BATCH_SIZE = 256  
 
print("Generare embeddings tagline...")
tagline_emb = sbert.encode(
    df['tagline'].tolist(),
    batch_size=BATCH_SIZE,
    show_progress_bar=True,
    convert_to_numpy=True
)
print(f"tagline_emb shape: {tagline_emb.shape}")
 
print("\nGenerare embeddings overview_summary...")
overview_emb = sbert.encode(
    df['overview_summary'].tolist(),
    batch_size=BATCH_SIZE,
    show_progress_bar=True,
    convert_to_numpy=True
)
print(f"overview_emb shape: {overview_emb.shape}")
 
print("\nGenerare embeddings review_summary...")
review_emb = sbert.encode(
    df['review_summary'].tolist(),
    batch_size=BATCH_SIZE,
    show_progress_bar=True,
    convert_to_numpy=True
)
print(f"review_emb shape: {review_emb.shape}")

np.save(OUTPUT_PATH + 'tagline_emb.npy', tagline_emb)
np.save(OUTPUT_PATH + 'overview_emb.npy', overview_emb)
np.save(OUTPUT_PATH + 'review_emb.npy', review_emb)
print("\nEmbeddings salvate!")
 
 

Generare embeddings tagline...


Batches:   0%|          | 0/158 [00:00<?, ?it/s]

tagline_emb shape: (40197, 384)

Generare embeddings overview_summary...


Batches:   0%|          | 0/158 [00:00<?, ?it/s]

overview_emb shape: (40197, 384)

Generare embeddings review_summary...


Batches:   0%|          | 0/158 [00:00<?, ?it/s]

review_emb shape: (40197, 384)

Embeddings salvate!


In [12]:
def evaluate_model(query_emb, doc_emb, pop_scores=None, pop_weight=0.0,
                   review_emb=None, review_weight=0.0,
                   batch_size=512, k_values=[1, 3, 5, 10]):
    """
    Evalueaza un model de recomandare prin Hit@K.
 
    Parametri:
        query_emb    : embeddings tagline (N, 384)
        doc_emb      : embeddings overview (N, 384)
        pop_scores   : popularitate normalizata (N,) — optional
        pop_weight   : ponderea popularitatii in scor final (0.0 - 1.0)
        review_emb   : embeddings review (N, 384) — optional
        review_weight: ponderea review-ului in scor final (0.0 - 1.0)
        batch_size   : cate query-uri procesam simultan
        k_values     : valorile K pentru Hit@K
 
    Returnează: dict cu Hit@K pentru fiecare K
    """
    N = len(query_emb)
    hits = {k: 0 for k in k_values}
    max_k = max(k_values)
 
    for start in range(0, N, batch_size):
        end = min(start + batch_size, N)
        batch_query = query_emb[start:end]  
 
        cos_overview = cosine_similarity(batch_query, doc_emb)
 
        overview_weight = 1.0 - pop_weight - review_weight
        score = overview_weight * cos_overview
 
        if review_weight > 0 and review_emb is not None:
            cos_review = cosine_similarity(batch_query, review_emb)
            score += review_weight * cos_review
 
        if pop_weight > 0 and pop_scores is not None:
            score += pop_weight * pop_scores[np.newaxis, :]
 
        top_k_indices = np.argpartition(score, -max_k, axis=1)[:, -max_k:]
 
        for i, global_idx in enumerate(range(start, end)):
            for k in k_values:
                top_k = top_k_indices[i][np.argsort(score[i][top_k_indices[i]])[::-1][:k]]
                if global_idx in top_k:
                    hits[k] += 1
 
        if start % 5000 == 0:
            print(f"  Procesat {start}/{N}...")
 
    return {k: hits[k] / N for k in k_values}
 

In [13]:
pop_scores = df['popularity_norm'].values
 
print("EVALUARE MODELE")
 
results = {}
 
print("\nM1: overview_summary (baseline)")
results['M1_overview'] = evaluate_model(
    query_emb=tagline_emb,
    doc_emb=overview_emb
)
print(f"  {results['M1_overview']}")
 
print("\nM2a: overview(0.9) + popularity(0.1)")
results['M2a_ov_pop_90_10'] = evaluate_model(
    query_emb=tagline_emb,
    doc_emb=overview_emb,
    pop_scores=pop_scores,
    pop_weight=0.1
)
print(f"  {results['M2a_ov_pop_90_10']}")
 

print("\nM2b: overview(0.7) + popularity(0.3)")
results['M2b_ov_pop_70_30'] = evaluate_model(
    query_emb=tagline_emb,
    doc_emb=overview_emb,
    pop_scores=pop_scores,
    pop_weight=0.3
)
print(f"  {results['M2b_ov_pop_70_30']}")
 
review_indices = df[df['review_summary'] != ''].index.tolist()
print(f"\nM3: overview(0.7) + review(0.3) — pe {len(review_indices):,} filme cu review")
results['M3_ov_review'] = evaluate_model(
    query_emb=tagline_emb[review_indices],
    doc_emb=overview_emb[review_indices],
    review_emb=review_emb[review_indices],
    review_weight=0.3
)
print(f"  {results['M3_ov_review']}")
 

pop_review = pop_scores[review_indices]
print(f"\nM4: overview(0.6) + review(0.3) + popularity(0.1) — pe {len(review_indices):,} filme")
results['M4_ov_review_pop'] = evaluate_model(
    query_emb=tagline_emb[review_indices],
    doc_emb=overview_emb[review_indices],
    pop_scores=pop_review,
    pop_weight=0.1,
    review_emb=review_emb[review_indices],
    review_weight=0.3
)
print(f"  {results['M4_ov_review_pop']}")
 
print(f"\nM1_subset: overview only — pe {len(review_indices):,} filme cu review (comparatie cu M3/M4)")
results['M1_subset'] = evaluate_model(
    query_emb=tagline_emb[review_indices],
    doc_emb=overview_emb[review_indices]
)
print(f"  {results['M1_subset']}")
 
 

EVALUARE MODELE

M1: overview_summary (baseline)
  Procesat 0/40197...
  {1: 0.05107346319377068, 3: 0.07570216682837028, 5: 0.08868821056297733, 10: 0.10873945816852004}

M2a: overview(0.9) + popularity(0.1)
  Procesat 0/40197...
  {1: 0.05060079110381372, 3: 0.07470706769161878, 5: 0.08781749881831978, 10: 0.10769460407493096}

M2b: overview(0.7) + popularity(0.3)
  Procesat 0/40197...
  {1: 0.027688633480110456, 3: 0.04609796751001319, 5: 0.05774062741000572, 10: 0.07580167674204542}

M3: overview(0.7) + review(0.3) — pe 11,644 filme cu review
  Procesat 0/11644...
  {1: 0.05608038474750945, 3: 0.08905874270010306, 5: 0.10795259361044314, 10: 0.13569220199244245}

M4: overview(0.6) + review(0.3) + popularity(0.1) — pe 11,644 filme
  Procesat 0/11644...
  {1: 0.05307454482995534, 3: 0.0863105462040536, 5: 0.10417382342837513, 10: 0.132944005496393}

M1_subset: overview only — pe 11,644 filme cu review (comparatie cu M3/M4)
  Procesat 0/11644...
  {1: 0.05616626588801099, 3: 0.0880281

In [14]:
print("\n")
print(f"{'Model':<35} {'Hit@1':>8} {'Hit@3':>8} {'Hit@5':>8} {'Hit@10':>8}")
 
model_labels = {
    'M1_overview':       'M1: overview (toate filmele)',
    'M2a_ov_pop_90_10':  'M2a: overview(0.9)+pop(0.1)',
    'M2b_ov_pop_70_30':  'M2b: overview(0.7)+pop(0.3)',
    'M1_subset':         'M1_sub: overview (subset review)',
    'M3_ov_review':      'M3: overview(0.7)+review(0.3)',
    'M4_ov_review_pop':  'M4: overview(0.6)+review(0.3)+pop(0.1)',
}
 
for key, label in model_labels.items():
    r = results[key]
    print(f"{label:<35} {r[1]:>7.3f} {r[3]:>8.3f} {r[5]:>8.3f} {r[10]:>8.3f}")
 
print("Nota: M3/M4/M1_sub evaluate pe ~11.6k filme cu review.")
print("      M1/M2 evaluate pe toate cele 40.2k filme.")
 



Model                                  Hit@1    Hit@3    Hit@5   Hit@10
M1: overview (toate filmele)          0.051    0.076    0.089    0.109
M2a: overview(0.9)+pop(0.1)           0.051    0.075    0.088    0.108
M2b: overview(0.7)+pop(0.3)           0.028    0.046    0.058    0.076
M1_sub: overview (subset review)      0.056    0.088    0.106    0.130
M3: overview(0.7)+review(0.3)         0.056    0.089    0.108    0.136
M4: overview(0.6)+review(0.3)+pop(0.1)   0.053    0.086    0.104    0.133
Nota: M3/M4/M1_sub evaluate pe ~11.6k filme cu review.
      M1/M2 evaluate pe toate cele 40.2k filme.
